# Credit Card Fraud Detection: Tackling Extreme Class Imbalance

## 1. Problem Overview
Credit card fraud costs global payment ecosystems tens of billions of dollars annually. Building machine learning systems for fraud detection presents distinct challenges:
- **Extreme Class Imbalance**: Genuine transactions drastically outnumber fraudulent transactions (~0.17% fraud rate).
- **Deceptive Accuracy**: A naive model predicting 100% genuine transactions scores 99.83% accuracy but fails 100% of fraud cases.
- **Asymmetric Cost of Errors**:
  - *False Negatives (Missed Fraud)*: Directly costs money (unrecovered stolen funds).
  - *False Positives (False Alarms)*: Leads to operational friction, card declines, and customer churn.

In this notebook, we explore data properties, scale non-PCA features without leakage, benchmark six imbalance treatment strategies across Logistic Regression, Random Forest, and XGBoost, and evaluate them on Precision-Recall AUC (PR-AUC) and operational financial cost.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is accessible
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.eda import load_data, inspect_basics
from src.model import prepare_data, build_models, evaluate_predictions
from src.evaluate import load_results

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)


## 2. Exploratory Data Analysis (EDA)
Let us load the dataset and inspect basic distributions, missing values, and class proportions.


In [ ]:
df = load_data(project_root / 'data' / 'creditcard.csv')
stats = inspect_basics(df)
df.head()


### 2.1 Transaction Amount & Time Dynamics
- Time: Elapsed seconds since the first transaction in the dataset (48-hour window).
- Amount: Highly right-skewed with extreme transactions exceeding ,000.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution by class
sns.boxplot(data=df, x='Class', y='Amount', showfliers=False, ax=axes[0], palette=['#2b5c8f', '#d9534f'])
axes[0].set_xticklabels(['Genuine (0)', 'Fraud (1)'])
axes[0].set_title('Transaction Amount Comparison (Excl. Outliers)', weight='bold')

# Hourly transaction volume
df['Hour'] = (df['Time'] / 3600) % 24
sns.kdeplot(df[df['Class'] == 0]['Hour'], label='Genuine', fill=True, alpha=0.3, ax=axes[1], color='#2b5c8f')
sns.kdeplot(df[df['Class'] == 1]['Hour'], label='Fraud', fill=True, alpha=0.4, ax=axes[1], color='#d9534f')
axes[1].set_title('Hourly Transaction Density (24-Hour Cycle)', weight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()


### 2.2 Feature Correlation with Fraud
Because features $ to {28}$ are PCA components, they are orthogonal to each other, but they have varying correlation with the target $.


In [ ]:
corr = df.corr()['Class'].drop(['Class', 'Hour']).sort_values()
top_neg = corr.head(5)
top_pos = corr.tail(5)
top_corr = pd.concat([top_neg, top_pos])

plt.figure(figsize=(10, 5))
colors = ['#2b5c8f' if v < 0 else '#d9534f' for v in top_corr.values]
plt.barh(top_corr.index, top_corr.values, color=colors, edgecolor='black')
plt.axvline(0, color='black', linestyle='--')
plt.title('Top 5 Negative and Positive Correlated Features with Class', weight='bold')
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()


## 3. Preprocessing & Leak-Free Scaling
To prevent data leakage:
1. We split into train (80%) and test (20%) using **Stratified Sampling** to lock the 0.17% fraud prevalence.
2. We fit RobustScaler on the training set only, and transform the test set using training parameters.


In [ ]:
X_train, X_test, y_train, y_test, scaler_amt, scaler_tm = prepare_data(
    data_path=str(project_root / 'data' / 'creditcard.csv'),
    test_size=0.2,
    random_state=42
)
print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)


## 4. Modeling & Strategy Comparison
We evaluate 6 distinct configurations:
1. **Logistic Regression (Baseline)**
2. **Logistic Regression (Class Weighted)**
3. **Logistic Regression + SMOTE** (Synthetic Minority Over-sampling)
4. **Random Forest (Balanced)**
5. **XGBoost (Weighted)** (scale_pos_weight = 577.3)
6. **XGBoost (Default)**


In [ ]:
from src.model import train_and_evaluate_all
results, curves = train_and_evaluate_all()


## 5. Visualizing Evaluation & Financial Cost Analysis


In [ ]:
from src.evaluate import (
    plot_pr_and_roc_curves,
    plot_metrics_comparison_bar,
    plot_confusion_matrices,
    plot_financial_cost_benefit
)

plot_pr_and_roc_curves(results, curves)
plot_metrics_comparison_bar(results)
plot_confusion_matrices(results)
plot_financial_cost_benefit(results)


## 6. Key Conclusions & Production Recommendations
1. **Tree Ensembles Outperform Linear Baselines**:
   - **Random Forest (Balanced)** achieved the highest **PR-AUC (0.8610)** and **F1-Score (0.8523)** with exceptional precision (96.15%) and only 3 false alarms out of 56,864 genuine transactions.
   - **XGBoost (Weighted)** delivered the highest fraud catch rate (**83.67% recall**) while retaining 73.87% precision.
2. **The Danger of Naive Oversampling**:
   - Applying SMOTE or class weighting to Logistic Regression elevated recall to 91.84%, but plunged precision down to ~5.9%, inundating fraud analysts with over 1,400 false alarms.
3. **Primary Metric Takeaway**:
   - For heavily skewed datasets, **PR-AUC** reflects actual deployment effectiveness far more reliably than ROC-AUC or standard accuracy.
